# exp072_exp063_full_replay_feature_cache train

CPU-only train feature cache generation for exp063-compatible full public replay PF/Beam/likelihood-PF features. This notebook does not train models and does not generate test features.

## Contents

1. Setup and configuration
2. Raw train input check
3. Train feature cache generation
4. Generated artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from feature_cache import run_train_feature_cache

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Feature cache variant:", cfg_get(config, "feature_cache.variant"))
print("Expected feature count:", cfg_get(config, "feature_cache.expected_feature_count"))
print("Kaggle GPU enabled:", cfg_get(config, "runtime.kaggle.enable_gpu"))


## 2. Raw train input check

In [ ]:
train_files = sorted(paths.train_data_dir.glob("*__horizontal_well.csv"))
typewell_files = sorted(paths.train_data_dir.glob("*__typewell.csv"))
print("Raw data dir:", paths.raw_data_dir)
print("Train dir:", paths.train_data_dir)
print("Horizontal wells:", len(train_files))
print("Typewells:", len(typewell_files))
print("First train files:", [path.name for path in train_files[:5]])


## 3. Train feature cache generation

In [ ]:
summary = run_train_feature_cache(
    data_dir=paths.raw_data_dir,
    output_dir=paths.artifacts_dir,
    n_jobs=int(cfg_get(config, "feature_cache.n_jobs", 8)),
    pf_seeds=int(cfg_get(config, "feature_cache.pf_seeds", 128)),
    pf_particles=int(cfg_get(config, "feature_cache.pf_particles", 500)),
    fast=bool(cfg_get(config, "feature_cache.fast", False)),
    max_wells=cfg_get(config, "feature_cache.max_wells"),
)
print(json.dumps(summary, indent=2))

expected = int(cfg_get(config, "feature_cache.expected_feature_count", 196))
actual = int(summary["feature_count"])
if actual != expected:
    raise ValueError(f"feature_count mismatch: {actual} != {expected}")


## 4. Generated artifacts

In [ ]:
schema_path = paths.artifacts_dir / "exp063_full_replay_feature_cache_feature_schema.csv"
summary_path = paths.artifacts_dir / "exp063_full_replay_feature_cache_summary.json"
feature_path = paths.artifacts_dir / "exp063_full_replay_feature_cache_pixiux_likpf_public_replay_train_features.csv.gz"

schema = pd.read_csv(schema_path)
display(schema.head())
print("Feature cache:", feature_path, "exists=", feature_path.exists())
print("Feature schema:", schema_path, "exists=", schema_path.exists())
print("Summary:", summary_path, "exists=", summary_path.exists())
